# 4. SentencePiece / Unigram

This topic is important because it introduces a slightly different way of thinking about tokenization.

So far:

- **BPE:** repeatedly merges frequent adjacent pairs.
- **WordPiece:** learns a subword vocabulary and tokenizes using greedy longest matching.
- **SentencePiece:** provides a framework that can apply **BPE or Unigram directly to raw text**.

The most important distinction is:

> **SentencePiece does not require you to split text into whitespace-separated words first.**

---

# 4.1 What is SentencePiece?

**SentencePiece** is a subword tokenizer/detokenizer designed to train directly from raw sentences. It supports both:

- **BPE**
- **Unigram Language Model**

The official SentencePiece documentation describes it as language-independent because it operates directly on raw Unicode text without requiring language-specific word segmentation.

### Conceptually

```text
Traditional pipeline

Raw text
   ↓
Word segmentation
   ↓
Subword tokenizer
   ↓
Tokens
```

versus:

```text
SentencePiece

Raw text
   ↓
SentencePiece
   ↓
Subword tokens
```

That difference becomes particularly useful for languages where **spaces do not reliably define words**.

---

# 4.2 Raw-Text Tokenization

Consider English:

```text
I love NLP
```

Spaces give us an obvious segmentation:

```text
I | love | NLP
```

But consider Chinese:

```text
我喜欢自然语言处理
```

There aren't spaces separating words in the same way.

Similarly, Japanese and Thai can present challenges for whitespace-based word segmentation.

SentencePiece can process the raw character stream directly rather than requiring a separate language-specific word tokenizer.

### Conceptually

```text
Raw text
我喜欢自然语言处理
        ↓
SentencePiece
        ↓
subword pieces
        ↓
token IDs
```

This is one reason SentencePiece became popular in **multilingual NLP**.

---

# 4.3 How Does SentencePiece Represent Spaces?

Here's a subtle but important detail.

SentencePiece represents whitespace using a special symbol:

```text
▁
```

This is U+2581, commonly called a lower one-eighth block.

For example:

```text
"I love NLP"
```

might contain pieces such as:

```text
["▁I", "▁love", "▁NLP"]
```

The `▁` tells the decoder that a space occurs before that piece.

### Why is this useful?

Suppose we simply produced:

```text
["I", "love", "NLP"]
```

The tokenizer would need additional rules to reconstruct:

```text
I love NLP
```

Instead:

```text
["▁I", "▁love", "▁NLP"]
```

contains the whitespace information.

Conceptually:

```text
▁I + ▁love + ▁NLP
          ↓
I love NLP
```

---

# 4.4 Language Independence

This is one of the major advantages of SentencePiece.

Traditional NLP pipelines often require language-specific preprocessing.

For example:

```text
English
→ whitespace-based segmentation

Japanese
→ Japanese morphological segmentation

Chinese
→ Chinese word segmentation
```

SentencePiece tries to avoid requiring these separate steps.

```text
English ─┐
Chinese ─┤
Japanese ├──→ SentencePiece → subwords
Thai ────┘
```

The tokenizer learns its vocabulary directly from the training text. The official documentation specifically highlights its usefulness for languages without explicit word boundaries, including Chinese, Japanese, and Thai.

### Interview point 🎯

If asked:

> **Why is SentencePiece useful for multilingual NLP?**

A strong answer is:

> It can train directly on raw text without requiring language-specific word segmentation, making the same general tokenization pipeline applicable across languages with different writing systems and word-boundary conventions.

---

# 4.5 SentencePiece Is Not the Same Thing as Unigram

This distinction is **very important**.

Students often say:

> "SentencePiece = Unigram."

❌ Not exactly.

SentencePiece is a **tokenization framework/library** that supports different subword algorithms, including:

```text
SentencePiece
├── BPE
└── Unigram
```

The official implementation explicitly supports both BPE and the Unigram language model.

So:

```text
SentencePiece
     │
     ├──────── BPE
     │
     └──────── Unigram
```

This is why your syllabus writes:

> **SentencePiece / Unigram**

---

# 4.6 SentencePiece + BPE

You already learned BPE.

With SentencePiece, the BPE algorithm can operate directly on raw text.

Conceptually:

```text
Raw text
   ↓
Normalize
   ↓
Represent spaces using ▁
   ↓
Basic symbols
   ↓
BPE merge operations
   ↓
Subword vocabulary
```

For example:

```text
"lower"
```

might eventually become:

```text
▁lower
```

if that complete piece is learned.

Or a rare word might become:

```text
▁low + er
```

The exact segmentation depends on the trained vocabulary.

### Important connection

**SentencePiece does not change the fundamental idea of BPE.**

It changes the **text-processing framework around it** so BPE can work directly on raw text.

---

# 4.7 The Unigram Model

Now comes the more interesting part.

**Unigram** takes a different approach from BPE.

### BPE

Starts small:

```text
characters
   ↓
merge
   ↓
larger pieces
   ↓
merge
   ↓
larger pieces
```

It is a **bottom-up merging process**.

### Unigram

Starts with a relatively large set of candidate pieces and **removes/prunes pieces** while optimizing a probabilistic language model.

Conceptually:

```text
Large candidate vocabulary
          ↓
Assign probabilities
          ↓
Evaluate vocabulary
          ↓
Remove low-value pieces
          ↓
Repeat
          ↓
Final vocabulary
```

This distinction is central to understanding Unigram.

---

# 4.8 Why "Unigram"?

The model assigns probabilities to possible subword pieces.

Suppose a word can be segmented in several ways:

```text
playing
```

Possible segmentations:

```text
play + ing
pla + ying
playing
p + lay + ing
```

The Unigram model assigns probabilities to possible pieces and evaluates possible segmentations.

We can think of:

$$
P(\text{segmentation}) = \prod_i P(t_i)
$$

where:

- $t_i$ = a token/subword piece
- $P(t_i)$ = probability assigned to that piece

For example:

```text
play + ing
```

$$
P(\text{play}) \times P(\text{ing})
$$

versus:

```text
playing
```

$$
P(\text{playing})
$$

The model can compare the likelihoods of different segmentations.

---

# 4.9 Choosing a Segmentation

Suppose:

```text
Input:
playing
```

and the vocabulary contains:

```text
play
##ing
playing
```

A Unigram model could consider:

```text
Segmentation A:
[playing]

Segmentation B:
[play, ing]
```

It evaluates the probability of these segmentations.

Conceptually:

$$
P(\text{playing})
=
P(\text{playing})
$$

versus:

$$
P(\text{play},\text{ing})
=
P(\text{play})P(\text{ing})
$$

The segmentation with higher probability is preferred when performing deterministic tokenization.

⚠️ This is a simplified explanation. The actual Unigram model and decoding procedure involve a vocabulary model and dynamic programming rather than simply comparing two manually selected alternatives.

---

# 4.10 Unigram vs BPE

This is the comparison you should memorize.

| | **BPE** | **Unigram** |
|---|---|---|
| Basic philosophy | Merge pieces | Remove/prune pieces |
| Starting point | Small units | Large candidate vocabulary |
| Main idea | Frequent pair merging | Probabilistic vocabulary |
| Tokenization | Apply learned merges | Find high-probability segmentation |
| Direction | Bottom-up | Vocabulary reduction |
| Probabilistic | Not fundamentally | **Yes** |
| SentencePiece support | ✅ | ✅ |

### Mental model

```text
BPE

small vocabulary
      ↓
    MERGE 🔗
      ↓
larger vocabulary
```

versus:

```text
Unigram

large candidate vocabulary
      ↓
    PRUNE ✂️
      ↓
smaller vocabulary
```

That's the easiest way to remember the difference.

---

# 4.11 Unigram Can Have Multiple Valid Segmentations

This is another important property.

Suppose:

```text
New York
```

can potentially be represented as:

```text
["▁New", "▁York"]
```

or:

```text
["▁New", "▁York"]
```

or other candidate segmentations depending on the learned vocabulary.

During **subword regularization**, SentencePiece can sample different valid segmentations instead of always producing exactly one segmentation. The official SentencePiece implementation supports this for Unigram and also provides BPE-Dropout for BPE.

This can act as a form of data augmentation:

```text
Same sentence
      │
      ├── segmentation A
      ├── segmentation B
      └── segmentation C
```

The model therefore sees slightly different tokenizations during training.

This is an **advanced concept** for now; just understand the idea.

---

# 4.12 Why SentencePiece Matters for Modern NLP

SentencePiece became particularly useful in multilingual and sequence-to-sequence systems.

Examples include:

- **T5**
- **ALBERT**
- various multilingual models

The exact tokenizer/model combination varies by architecture, so don't memorize:

> "Every modern model uses SentencePiece."

That's false.

Instead remember:

> **SentencePiece is a framework capable of training BPE or Unigram subword models directly from raw text.**

---

# 4.13 Important Correction: SentencePiece ≠ "One Tokenizer Algorithm"

This is worth emphasizing because it frequently causes confusion.

Think:

```text
                 SentencePiece
                      │
              ┌───────┴───────┐
              ↓               ↓
             BPE           Unigram
              │               │
         merge pairs      probabilistic
                          segmentation
```

So if someone says:

> "SentencePiece uses Unigram."

The technically better answer is:

> **SentencePiece supports Unigram and BPE; a particular model may use one of them.**

---

# 4.14 Connection to What We Already Learned

You now have three major approaches:

```text
BPE
 │
 └── Frequent pair → merge

WordPiece
 │
 └── Learn vocabulary
       ↓
       greedy longest match

SentencePiece
 │
 ├── BPE
 │    └── merge-based
 │
 └── Unigram
      └── probabilistic
          segmentation
```

And the raw-text advantage:

```text
                 RAW TEXT
                   │
         ┌─────────┴─────────┐
         ↓                   ↓
  Traditional          SentencePiece
  word segmentation          │
         ↓                   │
  subword tokenizer          ↓
                         BPE / Unigram
```

---

# Must Know ⭐

1. **SentencePiece can tokenize directly from raw text.**
2. It does **not require whitespace-based word segmentation**.
3. It represents spaces using the special symbol **`▁`**.
4. SentencePiece supports both:
   - **BPE**
   - **Unigram**
5. SentencePiece is useful for **language-independent/multilingual tokenization**.
6. **BPE = merge frequent pairs.**
7. **Unigram = probabilistic vocabulary + segmentation.**

### Good to Know

- SentencePiece tokenization is designed to be reversible/lossless with respect to its normalized input.
- Languages such as Chinese, Japanese, and Thai can benefit because explicit whitespace isn't required for word boundaries.
- Unigram can support **subword regularization**, where multiple segmentations may be sampled during training.

### Optional / Advanced

- Viterbi decoding for Unigram
- EM-style training of the Unigram language model
- Subword regularization
- BPE-Dropout
- SentencePiece normalization rules

---

## 🧠 The 20-second mental model

If I ask you in an interview:

> **"Explain SentencePiece and Unigram."**

You should be able to say:

> **SentencePiece is a tokenizer framework that can learn subword models directly from raw text without requiring language-specific word segmentation. It supports algorithms such as BPE and the Unigram language model. BPE builds vocabulary through pair merging, while Unigram learns a probabilistic vocabulary and selects likely segmentations. SentencePiece represents spaces using `▁`, which helps make tokenization reversible.**

That's a strong Module 25-level answer. 🎯

## What you should remember

1. **SentencePiece works directly on raw text.**
2. **`▁` represents whitespace.**
3. **It is not synonymous with Unigram.**
4. **SentencePiece supports both BPE and Unigram.**
5. **BPE builds by merging; Unigram builds by probabilistic vocabulary pruning/selection.**
6. **Unigram can consider multiple possible segmentations.**
7. This design is particularly useful for **multilingual and languages without explicit word boundaries**.

---